# Phân loại ảnh thời trang bằng CNN — Notebook Demo (dùng cho buổi vấn đáp)

Notebook này **kế thừa** notebook gốc `Fashion_MNIST_CNN.ipynb` (giữ nguyên, không chỉnh sửa) — copy phần khung sườn (mục 1-10), sau đó bổ sung mục 11 "Demo trực tiếp" để trình diễn khi vấn đáp: dự đoán ảnh ngẫu nhiên kèm biểu đồ độ tự tin, kiểm tra nhanh nhiều ảnh, và upload ảnh chụp thật để mô hình đoán thử.

> Chạy tuần tự từ trên xuống dưới (Runtime > Run all) trước, rồi vào buổi vấn đáp chỉ cần chạy lại các cell trong mục 11 nhiều lần để trình diễn — không cần train lại.


## 1. Setup môi trường

In [ ]:
import sys, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                     "torch", "torchvision", "scikit-learn", "seaborn", "matplotlib", "tqdm"])
    print("Colab detected: dependencies installed.")
else:
    print("Local environment detected: assuming requirements.txt already installed.")


In [ ]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

DATA_DIR = Path("data")
FIGURE_DIR = Path("figures")
FIGURE_DIR.mkdir(exist_ok=True)

CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]


## 2. Tải dữ liệu & Data Augmentation

Data augmentation (chỉ áp dụng cho tập train): lật ngang, xoay nhẹ, tịnh tiến nhẹ — giúp mô hình tổng quát hóa tốt hơn, giảm overfitting.

In [ ]:
MEAN, STD = (0.2860,), (0.3530,)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

BATCH_SIZE = 128
VAL_SPLIT = 0.1

full_train_aug = datasets.FashionMNIST(root=DATA_DIR, train=True, download=True, transform=train_transform)
full_train_eval = datasets.FashionMNIST(root=DATA_DIR, train=True, download=True, transform=eval_transform)
test_set = datasets.FashionMNIST(root=DATA_DIR, train=False, download=True, transform=eval_transform)

n_val = int(len(full_train_aug) * VAL_SPLIT)
n_train = len(full_train_aug) - n_val
generator = torch.Generator().manual_seed(SEED)
train_idx, val_idx = random_split(range(len(full_train_aug)), [n_train, n_val], generator=generator)

train_set = Subset(full_train_aug, train_idx.indices)
val_set = Subset(full_train_eval, val_idx.indices)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")


### Xem thử một số ảnh mẫu

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
raw_set = datasets.FashionMNIST(root=DATA_DIR, train=True, download=True)
for ax in axes.flat:
    idx = random.randrange(len(raw_set))
    img, label = raw_set[idx]
    ax.imshow(img, cmap="gray")
    ax.set_title(CLASS_NAMES[label], fontsize=8)
    ax.axis("off")
fig.tight_layout()
plt.show()


## 3. Định nghĩa mô hình

- **MLP**: baseline fully-connected, có BatchNorm + Dropout.
- **CNN**: 2 khối Conv (Conv-BN-ReLU x2 -> MaxPool -> Dropout), sau đó fully-connected có BatchNorm + Dropout.

In [ ]:
class MLP(nn.Module):
    def __init__(self, num_classes=10, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.net(x)


class CNN(nn.Module):
    def __init__(self, num_classes=10, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(dropout / 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(dropout / 2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


## 4. Vòng lặp huấn luyện / đánh giá

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)
    return total_loss / total, correct / total


def fit(model, train_loader, val_loader, epochs=20, lr=1e-3, weight_decay=1e-4):
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    progress = tqdm(range(1, epochs + 1), desc="Epochs")
    for epoch in progress:
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        progress.set_postfix(
            train_loss=f"{train_loss:.4f}", train_acc=f"{train_acc:.4f}",
            val_loss=f"{val_loss:.4f}", val_acc=f"{val_acc:.4f}",
        )
    return history


@torch.no_grad()
def predict(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for images, labels in loader:
        images = images.to(DEVICE)
        preds = model(images).argmax(1).cpu()
        all_preds.append(preds)
        all_labels.append(labels)
    return torch.cat(all_preds).numpy(), torch.cat(all_labels).numpy()


def plot_history(history, title, save_path=None):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["train_loss"], label="train")
    axes[0].plot(history["val_loss"], label="val")
    axes[0].set_title(f"{title} - Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
    axes[1].plot(history["train_acc"], label="train")
    axes[1].plot(history["val_acc"], label="val")
    axes[1].set_title(f"{title} - Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


def plot_confusion(y_true, y_pred, title, save_path=None):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(8, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


## Tái sử dụng mô hình đã train (không cần train lại mỗi lần mở notebook)

Notebook này tự động lưu mô hình đã train vào **Google Drive** (thư mục `MyDrive/BTL_AI_checkpoints/`). Lần đầu chạy, mô hình sẽ train bình thường rồi tự lưu lại. **Các lần chạy sau — kể cả mở notebook khác trong dự án này (Experiments, Demo) — sẽ tự động phát hiện đã có sẵn kết quả và load lại trong vài giây, thay vì train lại từ đầu (vốn mất vài phút mỗi lần).**

Khi chạy cell dưới, Colab sẽ hỏi xin quyền truy cập Google Drive — bấm **Allow** để tiếp tục.

Nếu muốn ép train lại từ đầu (ví dụ đổi kiến trúc/hyperparameter), xóa file tương ứng trong thư mục `MyDrive/BTL_AI_checkpoints/` trên Drive rồi chạy lại notebook.


In [ ]:
import json

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_CKPT_DIR = Path("/content/drive/MyDrive/BTL_AI_checkpoints")
else:
    DRIVE_CKPT_DIR = Path("checkpoints")  # chạy local: chỉ lưu trong thư mục hiện tại

DRIVE_CKPT_DIR.mkdir(parents=True, exist_ok=True)
print("Thư mục lưu/đọc checkpoint:", DRIVE_CKPT_DIR.resolve())


def load_or_train(model_class, name, loader, epochs, dropout=0.3):
    """Có checkpoint (weights + history) sẵn trên Drive -> load luôn, không train lại.
    Chưa có -> train mới rồi lưu lại lên Drive để các lần sau (kể cả notebook khác) dùng ngay."""
    ckpt_path = DRIVE_CKPT_DIR / f"{name}.pt"
    history_path = DRIVE_CKPT_DIR / f"{name}_history.json"

    if ckpt_path.exists() and history_path.exists():
        model = model_class(dropout=dropout)
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        model.to(DEVICE)
        with open(history_path) as f:
            history = json.load(f)
        print(f"[{name}] Tìm thấy checkpoint có sẵn trên Drive -> load lại, KHÔNG train lại.")
        return model, history

    print(f"[{name}] Chưa có checkpoint trên Drive -> bắt đầu train mới...")
    model = model_class(dropout=dropout)
    history = fit(model, loader, val_loader, epochs=epochs)
    torch.save(model.state_dict(), ckpt_path)
    with open(history_path, "w") as f:
        json.dump(history, f)
    print(f"[{name}] Đã train xong, lưu checkpoint vào {ckpt_path}")
    return model, history


## 5. Chuẩn bị mô hình MLP — tự động **load nếu đã có checkpoint trên Drive**, chỉ train nếu chưa có


In [ ]:
EPOCHS = 20

mlp_model, mlp_history = load_or_train(MLP, "mlp", train_loader, epochs=EPOCHS)
plot_history(mlp_history, "MLP", save_path=FIGURE_DIR / "mlp_history.png")


## 6. Chuẩn bị mô hình CNN — tự động **load nếu đã có checkpoint trên Drive**, chỉ train nếu chưa có


In [ ]:
cnn_model, cnn_history = load_or_train(CNN, "cnn", train_loader, epochs=EPOCHS)
plot_history(cnn_history, "CNN", save_path=FIGURE_DIR / "cnn_history.png")


## 7. Đánh giá trên tập test: Accuracy, Precision, Recall, F1-score, Confusion Matrix

In [ ]:
mlp_pred, mlp_true = predict(mlp_model, test_loader)
print("=== MLP classification report ===")
mlp_report = classification_report(mlp_true, mlp_pred, target_names=CLASS_NAMES, digits=4, output_dict=True)
print(classification_report(mlp_true, mlp_pred, target_names=CLASS_NAMES, digits=4))
plot_confusion(mlp_true, mlp_pred, "MLP Confusion Matrix", save_path=FIGURE_DIR / "mlp_confusion_matrix.png")


In [ ]:
cnn_pred, cnn_true = predict(cnn_model, test_loader)
print("=== CNN classification report ===")
cnn_report = classification_report(cnn_true, cnn_pred, target_names=CLASS_NAMES, digits=4, output_dict=True)
print(classification_report(cnn_true, cnn_pred, target_names=CLASS_NAMES, digits=4))
plot_confusion(cnn_true, cnn_pred, "CNN Confusion Matrix", save_path=FIGURE_DIR / "cnn_confusion_matrix.png")


## 8. So sánh CNN vs MLP

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Model": ["MLP", "CNN"],
    "Accuracy": [mlp_report["accuracy"], cnn_report["accuracy"]],
    "Precision (macro)": [mlp_report["macro avg"]["precision"], cnn_report["macro avg"]["precision"]],
    "Recall (macro)": [mlp_report["macro avg"]["recall"], cnn_report["macro avg"]["recall"]],
    "F1-score (macro)": [mlp_report["macro avg"]["f1-score"], cnn_report["macro avg"]["f1-score"]],
    "Params": [sum(p.numel() for p in mlp_model.parameters()), sum(p.numel() for p in cnn_model.parameters())],
})
comparison


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(mlp_history["val_acc"], label="MLP val_acc")
ax.plot(cnn_history["val_acc"], label="CNN val_acc")
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation Accuracy")
ax.set_title("MLP vs CNN - Validation Accuracy qua các epoch")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "mlp_vs_cnn_val_acc.png", dpi=150, bbox_inches="tight")
plt.show()


## 9. Lưu mô hình (tuỳ chọn)

Trên Colab, có thể mount Google Drive để lưu checkpoint lâu dài thay vì lưu vào runtime tạm thời.

> Lưu ý: mục "Tái sử dụng mô hình đã train" ở trên đã tự động lưu `mlp`/`cnn` (weights + history) vào Google Drive rồi. Cell dưới đây chỉ lưu thêm 1 bản local trong phiên Colab hiện tại (sẽ mất khi ngắt kết nối) — không bắt buộc chạy.


In [ ]:
CKPT_DIR = Path("checkpoints")
CKPT_DIR.mkdir(exist_ok=True)
torch.save(mlp_model.state_dict(), CKPT_DIR / "mlp.pt")
torch.save(cnn_model.state_dict(), CKPT_DIR / "cnn.pt")
print("Saved checkpoints to", CKPT_DIR.resolve())


## 10. Kết luận

Điền nhận xét dựa trên bảng so sánh và các biểu đồ ở trên, ví dụ:
- CNN có Accuracy/F1-score cao hơn MLP hay không, chênh lệch bao nhiêu?
- Data Augmentation, Dropout, Batch Normalization ảnh hưởng thế nào đến khoảng cách giữa train và val accuracy (overfitting)?
- Các lớp nào (class) bị nhầm lẫn nhiều nhất qua confusion matrix (ví dụ Shirt vs T-shirt/top vs Pullover vs Coat)?


---
# Phần bổ sung: Demo trực tiếp cho buổi vấn đáp

Mục 11 dưới đây không phục vụ phân tích số liệu như notebook Experiments, mà để **trình diễn trực quan**: bấm chạy là thấy ngay mô hình hoạt động, dễ giải thích cho người không rành AI, và ấn tượng với giám khảo.


## 11. Demo dự đoán trực tiếp

### 11.0 Hàm dùng chung: dự đoán 1 ảnh + tính độ tự tin (confidence)

Mô hình không chỉ đưa ra 1 nhãn duy nhất — nó tính ra **xác suất cho cả 10 loại**, rồi chọn loại có xác suất cao nhất làm câu trả lời. Hàm dưới đây lấy đủ 10 con số đó (gọi là **softmax probabilities**) để vẽ biểu đồ độ tự tin.


In [ ]:
import torch.nn.functional as F


def predict_with_confidence(model, image_tensor):
    """Trả về mảng 10 xác suất (softmax) cho 1 ảnh."""
    model.eval()
    with torch.no_grad():
        logits = model(image_tensor.unsqueeze(0).to(DEVICE))
        probs = F.softmax(logits, dim=1).cpu().numpy()[0]
    return probs


### 11.1 Dự đoán 1 ảnh ngẫu nhiên + biểu đồ độ tự tin

Chạy cell dưới nhiều lần (Ctrl+Enter lặp lại) để mỗi lần ra 1 ảnh ngẫu nhiên khác — rất hợp để "diễn" trực tiếp khi giám khảo yêu cầu xem thử.


In [ ]:
def show_prediction(model, dataset, idx, model_name="CNN"):
    image, true_label = dataset[idx]
    probs = predict_with_confidence(model, image)
    pred_label = int(probs.argmax())
    is_correct = pred_label == true_label
    status = "ĐÚNG" if is_correct else "SAI"
    color = "green" if is_correct else "crimson"

    fig, axes = plt.subplots(1, 2, figsize=(10, 4), gridspec_kw={"width_ratios": [1, 1.6]})

    axes[0].imshow(image.squeeze(), cmap="gray")
    axes[0].axis("off")
    axes[0].set_title(f"Nhãn thật: {CLASS_NAMES[true_label]}", fontsize=10)

    bar_colors = ["#4C72B0"] * 10
    bar_colors[pred_label] = color
    axes[1].barh(CLASS_NAMES, probs, color=bar_colors)
    axes[1].set_xlim(0, 1)
    axes[1].invert_yaxis()
    axes[1].set_xlabel("Độ tự tin (xác suất)")
    axes[1].set_title(
        f"{model_name} đoán: {CLASS_NAMES[pred_label]} ({probs[pred_label] * 100:.1f}%) — {status}",
        color=color, fontsize=11, fontweight="bold",
    )
    fig.tight_layout()
    plt.show()


random_idx = random.randrange(len(test_set))
show_prediction(cnn_model, test_set, random_idx, model_name="CNN")


**Cách đọc & cách giải thích khi vấn đáp:** bên trái là ảnh gốc và nhãn thật; bên phải là 10 thanh ngang — mỗi thanh là % "tự tin" của mô hình cho 1 loại trang phục, thanh dài nhất là câu trả lời cuối cùng (tô màu xanh nếu đúng, đỏ nếu sai). Nếu 1 thanh vượt trội hẳn các thanh còn lại, mô hình rất chắc chắn; nếu vài thanh gần bằng nhau, mô hình đang "phân vân" giữa các loại có hình dáng giống nhau (ví dụ Shirt và T-shirt/top) — đây là ý hay để trả lời câu hỏi "mô hình có tự tin không, sai ở đâu thì sai như thế nào".


### 11.2 Kiểm tra nhanh nhiều ảnh cùng lúc

Lấy ngẫu nhiên 9 ảnh, viền **xanh lá** = đoán đúng, viền **đỏ** = đoán sai — nhìn phát biết ngay tỉ lệ đúng/sai mà không cần đọc số. Chạy lại cell nhiều lần trong buổi vấn đáp để giám khảo thấy mô hình đúng hầu hết các lần.


In [ ]:
from matplotlib.patches import Rectangle

fig, axes = plt.subplots(3, 3, figsize=(9, 9))
sample_indices = random.sample(range(len(test_set)), 9)
n_correct = 0
for ax, idx in zip(axes.flat, sample_indices):
    image, true_label = test_set[idx]
    probs = predict_with_confidence(cnn_model, image)
    pred_label = int(probs.argmax())
    is_correct = pred_label == true_label
    n_correct += int(is_correct)

    ax.imshow(image.squeeze(), cmap="gray")
    ax.set_xticks([])
    ax.set_yticks([])
    border_color = "limegreen" if is_correct else "red"
    ax.add_patch(Rectangle((0, 0), 27, 27, linewidth=4, edgecolor=border_color, facecolor="none"))
    ax.set_title(f"{CLASS_NAMES[pred_label]} ({probs[pred_label] * 100:.0f}%)", fontsize=9, color=border_color)

fig.suptitle(f"CNN đoán đúng {n_correct}/9 ảnh ngẫu nhiên", fontsize=13)
fig.tight_layout()
plt.show()


**Cách đọc:** viền xanh lá = mô hình đoán đúng, viền đỏ = đoán sai. Tiêu đề mỗi ảnh là nhãn mô hình đoán kèm % tự tin. Đây là cách trình diễn nhanh gọn, trực quan nhất khi giám khảo hỏi "cho xem thử mô hình chạy thế nào" mà không cần giải thích số liệu phức tạp.


### 11.3 Upload ảnh chụp thật để mô hình đoán thử

Phần trình diễn ấn tượng nhất: tải lên 1 ảnh chụp thật (áo, quần, giày, túi...) — **hoàn toàn không nằm trong Fashion-MNIST** — và xem mô hình đoán là gì. Chỉ chạy được khi mở notebook trên Google Colab (cần quyền truy cập file để upload).

**Lưu ý quan trọng để chụp ảnh cho ra kết quả tốt:**
- Chụp 1 món đồ duy nhất, nền đơn sắc (trắng/đen), object nằm giữa khung hình.
- Mô hình chỉ học trên ảnh 28×28 xám rất đơn giản, nên ảnh thật nhiều chi tiết/màu sắc/nền phức tạp có thể khiến mô hình đoán sai — đây **không phải lỗi code**, mà là hạn chế tự nhiên của mô hình khi gặp dữ liệu khác xa lúc train (gọi là dữ liệu "ngoài phân phối" — out-of-distribution). Nếu bị hỏi trong vấn đáp, đây là câu trả lời chuẩn.


In [ ]:
from PIL import Image
import numpy as np


def preprocess_custom_image(pil_image):
    img = pil_image.convert("L").resize((28, 28))
    arr = np.array(img).astype("float32") / 255.0

    # Fashion-MNIST: nền tối (giá trị thấp), vật thể sáng.
    # Ảnh chụp thật thường ngược lại (nền sáng, vật thể tối) -> tự động đảo màu nếu nền sáng.
    if arr.mean() > 0.5:
        arr = 1.0 - arr

    tensor = torch.tensor(arr).unsqueeze(0)
    tensor = (tensor - MEAN[0]) / STD[0]
    return tensor, img


if IN_COLAB:
    from google.colab import files
    print("Chọn 1 ảnh quần áo/giày dép/túi... từ máy để upload (jpg/png):")
    uploaded = files.upload()
    filename = next(iter(uploaded))
    pil_image = Image.open(filename)
else:
    custom_path = "PATH_TO_YOUR_IMAGE.jpg"  # sửa đường dẫn ảnh trên máy local tại đây
    pil_image = Image.open(custom_path)

tensor, preview_img = preprocess_custom_image(pil_image)
probs = predict_with_confidence(cnn_model, tensor)
pred_label = int(probs.argmax())

fig, axes = plt.subplots(1, 2, figsize=(10, 4), gridspec_kw={"width_ratios": [1, 1.6]})
axes[0].imshow(preview_img, cmap="gray")
axes[0].set_title("Ảnh của bạn (đã xử lý về 28x28)")
axes[0].axis("off")
axes[1].barh(CLASS_NAMES, probs, color=["green" if i == pred_label else "#4C72B0" for i in range(10)])
axes[1].invert_yaxis()
axes[1].set_xlim(0, 1)
axes[1].set_xlabel("Độ tự tin (xác suất)")
axes[1].set_title(f"CNN đoán: {CLASS_NAMES[pred_label]} ({probs[pred_label] * 100:.1f}%)", fontweight="bold")
fig.tight_layout()
plt.show()


**Gợi ý khi vấn đáp:** nếu mô hình đoán đúng ảnh chụp thật → nhấn mạnh "mô hình tổng quát hóa tốt, không chỉ học thuộc ảnh Fashion-MNIST". Nếu đoán sai → đây là cơ hội thể hiện hiểu biết: giải thích do khác biệt phân phối dữ liệu (ảnh thật có nền/góc chụp/độ sáng khác ảnh train), và nêu hướng khắc phục (train thêm dữ liệu đa dạng hơn, hoặc dùng mô hình pretrained trên ảnh thật như ImageNet).
